# InterestRateEngine Test Suite

Comprehensive testing of the InterestRateEngine with real MCP blockchain data.
Tests rate calculation, reputation scoring, and market analysis with real-time data.

## Setup & Imports

In [ ]:
import sys
import asyncio
import json
from decimal import Decimal
from datetime import datetime, timedelta

In [ ]:
sys.path.append('/home/mpo/algorand-showcase/algorand-lending-ecosystem/algorand-lending-business-logic')
from algorand_lending_bl.interest_rates import InterestRateEngine
from algorand_lending_bl.models import AlgorandAddress, RiskTier
from algorand_lending_bl.config import InterestRateConfig

In [ ]:
import httpx
# MCP Service endpoints
READER_URL = "http://localhost:8002"
MARKET_DATA_URL = "http://localhost:8789"

## Test Configuration

In [ ]:
# Test borrower addresses
prime_borrower = "7ZUECA7HFLZTXENRV24SHLU4AVPUTMTTDUFUBNBD64C5S3XM5THAIOF6Q"
standard_borrower = "GD64YIY3ZYQKECYR5736IALMJK2SYQHJNVK5UVSMP6JBTCDBBHU5A"
test_loan_amount = Decimal('1000')  # $1000 loan

In [ ]:
# Loan parameters
loan_duration = 90  # 90 days
collateral_assets = ["ALGO", "USDC"]
print(f"🎯 Test Parameters: ${test_loan_amount}, {loan_duration} days, {len(collateral_assets)} assets")

## Initialize InterestRateEngine

In [ ]:
# Create engine with default config
rate_engine = InterestRateEngine()
print(f"✅ InterestRateEngine initialized")
print(f"📊 Base Rate: {rate_engine.config.base_rate_percentage}%")

## Real Network Data Integration

In [ ]:
async def get_network_metrics():
    """Get real Algorand network metrics"""
    try:
        async with httpx.AsyncClient() as client:
            resp = await client.post(f"{READER_URL}/tools/get_health")
            return resp.json()
    except:
        return {"round": 0, "time-since-last-round": 0}

In [ ]:
# Fetch real network data
network_data = await get_network_metrics()
current_round = network_data.get("round", 0)
block_time = network_data.get("time-since-last-round", 4500)
print(f"🌐 Network: Round {current_round}, Block time: {block_time}ms")

## Borrower Reputation Analysis

In [ ]:
async def get_account_history(address: str, limit: int = 100):
    """Get borrower transaction history"""
    try:
        async with httpx.AsyncClient() as client:
            resp = await client.post(f"{READER_URL}/tools/search_transactions", 
                                   json={"address": address, "limit": limit})
            return resp.json().get("transactions", [])
    except:
        return []

In [ ]:
# Analyze prime borrower history
prime_history = await get_account_history(prime_borrower)
prime_tx_count = len(prime_history)
print(f"👑 Prime Borrower: {prime_tx_count} transactions")

In [ ]:
# Analyze standard borrower history
standard_history = await get_account_history(standard_borrower)
standard_tx_count = len(standard_history)
print(f"📊 Standard Borrower: {standard_tx_count} transactions")

## Market Rate Data

In [ ]:
async def get_defi_yields():
    """Get current DeFi yield rates"""
    try:
        async with httpx.AsyncClient() as client:
            resp = await client.post(f"{MARKET_DATA_URL}/tools/get_defi_data", 
                                   json={"protocol": "algorand"})
            return resp.json().get("apy", 5.0)
    except:
        return 5.0  # 5% fallback

In [ ]:
# Get current market rates
market_apy = await get_defi_yields()
algo_staking_rate = 6.2  # Current ALGO governance rate
print(f"📈 Market Rates: DeFi APY {market_apy}%, ALGO Staking {algo_staking_rate}%")

## Basic Rate Calculation Test

In [ ]:
# Calculate rate for prime borrower
prime_rate = await rate_engine.calculate_rate(
    borrower=prime_borrower,
    loan_amount_usd=test_loan_amount,
    loan_duration_days=loan_duration,
    collateral_assets=collateral_assets
)

In [ ]:
print(f"👑 Prime Borrower Rate:")
print(f"   Final Rate: {prime_rate.final_rate_percentage:.2f}%")
print(f"   Base Rate: {prime_rate.base_rate:.2f}%")
print(f"   Risk Premium: {prime_rate.risk_premium:.2f}%")

## Risk Tier Comparison

In [ ]:
# Calculate rate for standard borrower
standard_rate = await rate_engine.calculate_rate(
    borrower=standard_borrower,
    loan_amount_usd=test_loan_amount,
    loan_duration_days=loan_duration,
    collateral_assets=collateral_assets
)

In [ ]:
print(f"📊 Standard Borrower Rate:")
print(f"   Final Rate: {standard_rate.final_rate_percentage:.2f}%")
print(f"   Risk Premium: {standard_rate.risk_premium:.2f}%")
rate_difference = standard_rate.final_rate_percentage - prime_rate.final_rate_percentage
print(f"   Premium vs Prime: +{rate_difference:.2f}%")

## Loan Amount Sensitivity

In [ ]:
# Test different loan amounts
loan_amounts = [Decimal('100'), Decimal('1000'), Decimal('10000')]
amount_rates = []
for amount in loan_amounts:
    rate = await rate_engine.calculate_rate(prime_borrower, amount, loan_duration, collateral_assets)
    amount_rates.append((amount, rate.final_rate_percentage))

In [ ]:
print(f"💰 Loan Amount Sensitivity:")
for amount, rate in amount_rates:
    print(f"   ${amount}: {rate:.2f}%")
rate_spread = max([r[1] for r in amount_rates]) - min([r[1] for r in amount_rates])
print(f"   Rate Spread: {rate_spread:.2f}%")

## Duration Risk Analysis

In [ ]:
# Test different loan durations
durations = [30, 90, 180, 365]  # days
duration_rates = []
for duration in durations:
    rate = await rate_engine.calculate_rate(prime_borrower, test_loan_amount, duration, collateral_assets)
    duration_rates.append((duration, rate.final_rate_percentage))

In [ ]:
print(f"⏰ Duration Risk Analysis:")
for duration, rate in duration_rates:
    print(f"   {duration} days: {rate:.2f}%")
term_premium = duration_rates[-1][1] - duration_rates[0][1]
print(f"   Term Premium (1yr vs 1mo): {term_premium:.2f}%")

## Collateral Asset Impact

In [ ]:
# Test different collateral compositions
collateral_scenarios = [
    (["ALGO"], "ALGO Only"),
    (["USDC"], "USDC Only"),
    (["ALGO", "USDC"], "Mixed Portfolio")
]

In [ ]:
print(f"🔒 Collateral Impact Analysis:")
for assets, name in collateral_scenarios:
    rate = await rate_engine.calculate_rate(prime_borrower, test_loan_amount, loan_duration, assets)
    print(f"   {name}: {rate.final_rate_percentage:.2f}%")

## Real-Time Market Adjustment

In [ ]:
async def get_algo_price_volatility():
    """Get ALGO price volatility data"""
    try:
        async with httpx.AsyncClient() as client:
            resp = await client.post(f"{MARKET_DATA_URL}/tools/get_historical_prices", 
                                   json={"symbol": "ALGO", "days": 30})
            return resp.json().get("volatility", 0.25)
    except:
        return 0.25  # 25% default volatility

In [ ]:
# Factor in market volatility
algo_volatility = await get_algo_price_volatility()
volatility_adjustment = min(algo_volatility * 10, 5.0)  # Cap at 5%
print(f"📊 Market Volatility: {algo_volatility:.1%}, Rate Adjustment: +{volatility_adjustment:.2f}%")

## Rate Components Breakdown

In [ ]:
# Detailed rate analysis
detailed_rate = await rate_engine.calculate_rate(
    borrower=prime_borrower,
    loan_amount_usd=test_loan_amount,
    loan_duration_days=loan_duration,
    collateral_assets=collateral_assets
)

In [ ]:
print(f"🔍 Rate Components Breakdown:")
print(f"   Base Rate: {detailed_rate.base_rate:.2f}%")
print(f"   Credit Risk Premium: {detailed_rate.credit_risk_adjustment:.2f}%")
print(f"   Liquidity Premium: {detailed_rate.liquidity_adjustment:.2f}%")
print(f"   Market Risk: {detailed_rate.market_risk_adjustment:.2f}%")
print(f"   Final Rate: {detailed_rate.final_rate_percentage:.2f}%")

## Performance Benchmarking

In [ ]:
import time
# Benchmark rate calculation speed
start_time = time.time()
for _ in range(5):
    await rate_engine.calculate_rate(prime_borrower, test_loan_amount, loan_duration, collateral_assets)
elapsed = time.time() - start_time

In [ ]:
print(f"⚡ Performance Benchmark:")
print(f"   5 calculations in {elapsed:.3f}s")
print(f"   Average: {elapsed/5:.3f}s per calculation")
print(f"   Rate: {5/elapsed:.1f} calculations/second")

## Stress Testing

In [ ]:
# Edge case testing
stress_scenarios = [
    {"amount": Decimal('0.01'), "name": "Micro Loan"},
    {"amount": Decimal('1000000'), "name": "Mega Loan"},
    {"duration": 1, "name": "1 Day Loan"},
    {"duration": 3650, "name": "10 Year Loan"}
]

In [ ]:
print("🧪 Stress Test Results:")
for scenario in stress_scenarios:
    try:
        amount = scenario.get("amount", test_loan_amount)
        duration = scenario.get("duration", loan_duration)
        rate = await rate_engine.calculate_rate(prime_borrower, amount, duration, collateral_assets)
        print(f"   {scenario['name']}: ✅ {rate.final_rate_percentage:.2f}%")
    except Exception as e:
        print(f"   {scenario['name']}: ❌ {str(e)[:50]}")

## Test Results Summary

In [ ]:
# Test summary
summary = {
    "timestamp": datetime.now().isoformat(),
    "network_data_fetched": current_round > 0,
    "market_rates_loaded": market_apy > 0,
    "prime_rate_calculated": prime_rate is not None,
    "rate_spread_reasonable": 0.5 <= rate_difference <= 5.0
}

In [ ]:
print("📋 InterestRateEngine Test Summary:")
for key, value in summary.items():
    status = "✅" if value else "❌"
    print(f"   {key}: {status} {value}")
print(f"\n🎯 InterestRateEngine Test Suite: {'PASSED' if all(summary.values()) else 'FAILED'}")